# Laboratorio: De Parquet a Delta Lake

---

**Magister en Data Science - Universidad del Desarrollo**  
**Curso:** Big Data y Cloud Computing  
**Módulo 3:** Procesamiento Distribuido con Apache Spark  
**Sesión 3:** Viernes 29 de mayo de 2026
**Docente:** Luis Castillo Faune

---

## Objetivo

Al finalizar este laboratorio serás capaz de:

- Medir empíricamente la diferencia entre transformaciones **Narrow** y **Wide** en Spark.
- Construir tu primer pipeline PySpark real: Bronze -> Silver con validacion de calidad.
- Identificar los **Shuffles** en un plan de ejecucion usando `.explain(True)`.
- Escribir datos en **Delta Lake** y consultar versiones anteriores con **time travel**.
- Conectar la experiencia con las decisiones de la **Fase 1 del proyecto**.

---

## Estructura

| Fase | Actividad |
|---|---|
| 1 | Setup, calentamiento y Narrow vs Wide |
| 2 | Pipeline Bronze->Silver con Delta Lake |
| 3 | Reflexion y conexion con el proyecto |
| 4 | Cierre |

---

## Dataset: NYC Yellow Taxi enero 2023
Fuente: [NYC Taxi & Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page#)

| Característica | Valor |
|---|---|
| Filas aprox. | ~3.000.000 |
| Formato de entrada | Parquet |
| Formato de salida | Delta Lake (ACID + time travel) |

---
# FASE 1: Setup y calentamiento
## Sección 0: Instalación

Ejecuta esta celda primero. Tarda aproximadamente 2 minutos.

> Si aparece un error, espera que termine y **reinicia el runtime**: `Entorno de ejecución -> Reiniciar sesión`.

In [1]:
!pip install pyspark==3.5.1 delta-spark==3.2.0 -q
print('Dependencias instaladas correctamente')


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Dependencias instaladas correctamente


## Sección 1: SparkSession con soporte Delta Lake

Usaremos `configure_spark_with_delta_pip` para configurar Delta Lake.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
import time, os, urllib.request, warnings
warnings.filterwarnings('ignore')

builder = (
    SparkSession.builder
    .appName('Lab-BigData-BronzeToSilver')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    # Configuracion obligatoria para Delta Lake
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print(f'SparkSession inicializada')
print(f'  Version de Spark:  {spark.version}')
print(f'  Modo:              {spark.sparkContext.master}')
print(f'  Cores disponibles: {spark.sparkContext.defaultParallelism}')
print()
print('Delta Lake activo')

ModuleNotFoundError: No module named 'pyspark'

## Sección 2: Descarga del dataset y capa Bronze

Descargamos el Parquet de NYC Taxi. Esta descarga simula la **ingesta** desde una fuente externa hacia la capa Bronze.

In [ ]:
URL_DATASET = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet'
ARCHIVO_RAW = 'taxi_nyc_enero_2023.parquet'
RUTA_BRONZE = 'datos/bronze/taxi'
RUTA_SILVER = 'datos/silver/taxi_delta'

os.makedirs('datos/bronze', exist_ok=True)
os.makedirs('datos/silver', exist_ok=True)

if os.path.exists(ARCHIVO_RAW):
    print(f'Dataset ya descargado: {ARCHIVO_RAW}')
else:
    print('Descargando dataset NYC Taxi enero 2023...')
    t0 = time.time()
    urllib.request.urlretrieve(URL_DATASET, ARCHIVO_RAW)
    tam_mb = os.path.getsize(ARCHIVO_RAW) / 1e6
    print(f'Descarga completada en {time.time()-t0:.1f}s - {tam_mb:.1f} MB')

print()
print('Cargando en Spark como capa Bronze...')
t0 = time.time()

# Bronze: datos crudos con metadata de ingesta
df_bronze = (
    spark.read.parquet(ARCHIVO_RAW)
    .withColumn('_ingestion_ts', F.current_timestamp())
    .withColumn('_source_file',  F.lit(ARCHIVO_RAW))
)

n_bronze = df_bronze.count()
print(f'Capa Bronze: {n_bronze:,} filas en {time.time()-t0:.2f}s')
print(f'Columnas: {len(df_bronze.columns)}')
df_bronze.printSchema()

## Sección 3a: Calentamiento - primeras operaciones PySpark

In [ ]:
# Vista previa
print('Primeros 5 registros de la capa Bronze:')
df_bronze.select('tpep_pickup_datetime','passenger_count','trip_distance','fare_amount','payment_type').show(5)

In [ ]:
# Estadisticas descriptivas
print('Estadisticas descriptivas:')
df_bronze.select('passenger_count','trip_distance','fare_amount','tip_amount').describe().show()

In [ ]:
# Distribucion de tipos de pago
print('Distribucion por tipo de pago en capa Bronze:')
df_bronze.groupBy('payment_type').count().orderBy('count', ascending=False).show()

n_invalidos = df_bronze.filter((F.col('fare_amount') <= 0) | (F.col('trip_distance') <= 0)).count()
print(f'Registros con fare_amount <= 0 o trip_distance <= 0: {n_invalidos:,} ({n_invalidos/n_bronze*100:.2f}%)')

## Sección 3b: Narrow vs. Wide - medición empírica

Antes de construir el pipeline, medimos la diferencia de tiempo entre una transformación
**Narrow** (sin Shuffle) y una **Wide** (con Shuffle) sobre los mismos 3 millones de filas.

La clase teórica definio estos conceptos. Ahora los veremos aplicados a datos.

| Tipo | Qué hace Spark | Costo |
|---|---|---|
| **Narrow** | Cada partición se procesa localmente, sin mover datos entre nodos | Bajo |
| **Wide** | Las particiones intercambian datos entre Executors (Shuffle: red + disco) | Alto |

In [ ]:
# Comparacion empirica: Narrow vs Wide
print('Comparacion: Narrow vs Wide sobre', f'{n_bronze:,}', 'filas')
print('=' * 52)

# NARROW: .filter()
# Cada particion se filtra localmente, sin ningun movimiento de datos
# entre Executors. Costo de red: cero.
print('NARROW: .filter(fare_amount > 0)')
print('  Cada particion se procesa localmente - sin Shuffle')
t0 = time.time()
n_filtrado = df_bronze.filter('fare_amount > 0').count()   # count() = accion
TIEMPO_NARROW = time.time() - t0
print(f'  Resultado: {n_filtrado:,} filas')
print(f'  Tiempo:    {TIEMPO_NARROW:.2f}s')
print()

# WIDE: .groupBy()
# Para agrupar por payment_type, todos los registros con el mismo valor
# deben estar en el mismo Executor. Spark los redistribuye -> SHUFFLE
print('WIDE: .groupBy(payment_type).count()')
print('  Spark redistribuye datos entre Executors -> SHUFFLE (red + disco)')
t0 = time.time()
r_wide = df_bronze.groupBy('payment_type').count().collect()  # collect() = accion
TIEMPO_WIDE = time.time() - t0
print(f'  Resultado: {len(r_wide)} grupos')
print(f'  Tiempo:    {TIEMPO_WIDE:.2f}s')
print()

factor = TIEMPO_WIDE / TIEMPO_NARROW if TIEMPO_NARROW > 0 else 0
print('=' * 52)
print(f'  Narrow (.filter):  {TIEMPO_NARROW:.2f}s')
print(f'  Wide (.groupBy):   {TIEMPO_WIDE:.2f}s')
print(f'  Factor:            {factor:.1f}x mas lento')
print('=' * 52)
print()
print('NOTA: En Colab (modo local) el impacto es menor porque no hay red real.')
print('Incluso una transformacion Wide podria ser mas rapida.')
print('En un cluster productivo con TBs de datos, este factor puede ser 100x.')

**Escribe aqui tu observación:**

> _(doble clic para editar)_
>
> **El Wide fue X veces más lento porque:**  ...

---
# FASE 2: Pipeline Bronze -> Silver

## Sección 4: Validación y reglas de calidad

In [ ]:
print('Aplicando reglas de calidad de datos...')
print()

checks = [
    ('fare_amount > 0',         df_bronze.filter(F.col('fare_amount') <= 0).count()),
    ('trip_distance > 0',       df_bronze.filter(F.col('trip_distance') <= 0).count()),
    ('passenger_count entre 1-8', df_bronze.filter((F.col('passenger_count') <= 0) | (F.col('passenger_count') > 8)).count()),
    ('pickup < dropoff',        df_bronze.filter(F.col('tpep_pickup_datetime') >= F.col('tpep_dropoff_datetime')).count()),
]

for regla, n_desc in checks:
    print(f'  Regla {regla}: {n_desc:>8,} registros descartados')

n_validos = df_bronze.filter(
    (F.col('fare_amount') > 0) &
    (F.col('trip_distance') > 0) &
    (F.col('passenger_count').between(1, 8)) &
    (F.col('tpep_pickup_datetime') < F.col('tpep_dropoff_datetime'))
).count()

print()
print(f'  Total Bronze:    {n_bronze:>10,}')
print(f'  Estimado Silver: {n_validos:>10,} ({n_validos/n_bronze*100:.1f}% retencion)')

**Escribe aquí:**

> *El porcentaje de retención es aceptable / no aceptable porque ...*

## Sección 5a: Transformaciones Silver (todas lazy)

Ninguna de estas lineas ejecuta nada: solo construyen el DAG. La ejecución ocurre en la Sección 6.

In [ ]:
print('Construyendo plan Silver (evaluacion lazy)...')
t0 = time.time()

df_silver = (
    df_bronze
    # Filtros de calidad (NARROW)
    .filter(
        (F.col('fare_amount')   > 0) &
        (F.col('trip_distance') > 0) &
        (F.col('passenger_count').between(1, 8)) &
        (F.col('tpep_pickup_datetime') < F.col('tpep_dropoff_datetime'))
    )
    # Columnas de tiempo derivadas (NARROW)
    .withColumn('duracion_min',
        F.round((F.unix_timestamp(F.col('tpep_dropoff_datetime')) -
                 F.unix_timestamp(F.col('tpep_pickup_datetime'))) / 60, 1))
    .withColumn('hora_inicio',   F.hour('tpep_pickup_datetime'))
    .withColumn('dia_semana',    F.dayofweek('tpep_pickup_datetime'))
    .withColumn('es_fin_semana', F.col('dia_semana').isin([1, 7]))
    # Columnas de tarifa (NARROW)
    .withColumn('propina_pct',
        F.round(F.col('tip_amount') / F.col('fare_amount') * 100, 1))
    .withColumn('tarifa_por_milla',
        F.round(F.col('fare_amount') / F.col('trip_distance'), 2))
    # Filtro adicional (NARROW)
    .filter((F.col('duracion_min') > 0) & (F.col('duracion_min') < 180))
    # Introduciendo una transformacion WIDE explicita: repartition por tipo de pago
    #.repartition(F.col('payment_type')) # Esto forzara un Shuffle (Exchange)
    # Seleccion de columnas Silver (NARROW)
    .select(
        'tpep_pickup_datetime', 'tpep_dropoff_datetime',
        'passenger_count', 'trip_distance', 'payment_type',
        'fare_amount', 'tip_amount', 'total_amount',
        'duracion_min', 'hora_inicio', 'dia_semana', 'es_fin_semana',
        'propina_pct', 'tarifa_por_milla',
        '_ingestion_ts', '_source_file'
    )
)

print(f'Plan construido en {time.time()-t0:.4f}s')
print('(Ninguna transformacion se ejecuto todavia - evaluacion lazy)')
df_silver.printSchema()

## Sección 5b: Identificar Shuffles con .explain(True)

Antes de escribir, veamos el plan de ejecución. Cada nodo `Exchange` es un **Shuffle**.
Este plan muestra lo que vimos en el Bloque C de la parte teórica.

In [ ]:
# Plan de ejecucion del DataFrame Silver
# Busca los nodos 'Exchange' en el output = cada uno es un Shuffle
print('Plan de ejecucion del DataFrame Silver:')
print('Cada nodo Exchange = un Shuffle (datos viajan entre Executors)')
print('=' * 60)
df_silver.explain(True)
print('=' * 60)
print()
print('Identifica en el plan:')
print('  NARROW - sin Exchange: Filter, Project, Scan')
print('         -> se ejecutan localmente por particion')
print('  WIDE   - con Exchange: si aparece en el plan')
print('         -> datos viajan por la red entre Executors')
print()
print('El pipeline Bronze->Silver tiene principalmente operaciones NARROW.')
print('El Shuffle controlado ocurre en la escritura particionada (.partitionBy).')

## Sección 6: Escritura en Delta Lake

Esta es la primera **acción** del pipeline. Aquí Spark ejecuta el DAG completo.

> La escritura con `.partitionBy()` genera un Shuffle **controlado** y **necesario**: organiza los datos por `payment_type` para que las consultas futuras lean solo las particiones relevantes.

In [ ]:
print('Escribiendo capa Silver en Delta Lake...')
print('(Esta celda dispara la ejecucion del DAG completo)')
print()

t0 = time.time()
(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .partitionBy('payment_type')
    .save(RUTA_SILVER)
)
TIEMPO_PIPELINE = time.time() - t0

# Verificar resultado
df_s = spark.read.format('delta').load(RUTA_SILVER)
n_silver = df_s.count()

print(f'Capa Silver escrita exitosamente en Delta Lake')
print(f'  Ruta:               {RUTA_SILVER}')
print(f'  Tiempo de pipeline: {TIEMPO_PIPELINE:.2f}s')
print()
print(f'  Filas Bronze:       {n_bronze:>10,}')
print(f'  Filas Silver:       {n_silver:>10,}')
print(f'  Descartados:        {n_bronze-n_silver:>10,} ({(n_bronze-n_silver)/n_bronze*100:.2f}%)')
print()

# Estructura en disco
print('Estructura de archivos en disco:')
for root, dirs, files in os.walk(RUTA_SILVER):
    level = root.replace(RUTA_SILVER, '').count(os.sep)
    indent = '  ' * level
    if level < 3:
        print(f'  {indent}{os.path.basename(root)}/')
        if level == 1:
            pq = [f for f in files if f.endswith('.parquet')]
            if pq: print(f'  {indent}  ({len(pq)} archivos Parquet)')

## Sección 7: Explorar la capa Silver

Ahora leeremos la capa Silver desde Delta Lake y haremos análisis sobre datos limpios.

In [ ]:
print('Analisis sobre la capa Silver:')
print()

print('Tarifa y propina promedio por tipo de pago:')
(
    df_s.groupBy('payment_type')
    .agg(
        F.count('*').alias('viajes'),
        F.round(F.avg('fare_amount'), 2).alias('tarifa_prom'),
        F.round(F.avg('propina_pct'), 1).alias('propina_pct_prom'),
        F.round(F.avg('duracion_min'), 1).alias('duracion_prom_min'),
    )
    .orderBy('viajes', ascending=False)
).show()

print('Top 5 horas con mas viajes:')
df_s.groupBy('hora_inicio').count().orderBy('count', ascending=False).limit(5).show()

print('Propina promedio: fin de semana vs semana:')
df_s.groupBy('es_fin_semana').agg(
    F.count('*').alias('viajes'),
    F.round(F.avg('propina_pct'), 2).alias('propina_pct_prom')
).show()

---
# FASE 3: Time Travel en Delta Lake

## Seccion 8: Historial de transacciones

Delta Lake mantiene un **transaction log** con el historial completo de escrituras. Como un `git log` para datos.

In [ ]:
print('Historial de transacciones de la tabla Delta:')
DeltaTable.forPath(spark, RUTA_SILVER).history().select('version','timestamp','operation','operationMetrics').show(truncate=False)

In [ ]:
# Simular segunda carga (append)
print('Simulando segunda carga de datos (append de 50.000 registros)...')
(
    df_silver.limit(50000)
    .write.format('delta').mode('append')
    .partitionBy('payment_type')
    .save(RUTA_SILVER)
)
print('Segunda carga completada')
print()
print('Historial actualizado:')
DeltaTable.forPath(spark, RUTA_SILVER).history().select('version','timestamp','operation').show()

In [ ]:
# Time travel
print('Time Travel en accion:')
print()

df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(RUTA_SILVER)
df_v1 = spark.read.format('delta').option('versionAsOf', 1).load(RUTA_SILVER)
df_actual = spark.read.format('delta').load(RUTA_SILVER)

n_v0, n_v1, n_actual = df_v0.count(), df_v1.count(), df_actual.count()

print(f'  Version 0 (carga inicial):    {n_v0:>10,} filas')
print(f'  Version 1 (despues del append): {n_v1:>8,} filas')
print(f'  Version actual:               {n_actual:>10,} filas')
print()

avg_v0 = df_v0.agg(F.round(F.avg('fare_amount'),2)).collect()[0][0]
avg_v1 = df_v1.agg(F.round(F.avg('fare_amount'),2)).collect()[0][0]
print(f'  Tarifa promedio v0: ${avg_v0}')
print(f'  Tarifa promedio v1: ${avg_v1}')
print()
print('El time travel permite:')
print('  -> Auditar cambios en los datos')
print('  -> Reentrenar modelos ML con snapshots historicos')
print('  -> Recuperarse de una escritura incorrecta (rollback)')

In [ ]:
# Spark SQL sobre Delta
df_v0.createOrReplaceTempView('silver_v0')
print('Consulta SQL directa sobre la capa Silver (version 0):')
spark.sql("""
    SELECT
        CASE payment_type
            WHEN 1 THEN 'Tarjeta'
            WHEN 2 THEN 'Efectivo'
            WHEN 3 THEN 'Sin cargo'
            WHEN 4 THEN 'Disputa'
            ELSE 'Otro'
        END AS tipo_pago,
        COUNT(*) AS viajes,
        ROUND(AVG(fare_amount), 2) AS tarifa_prom,
        ROUND(AVG(propina_pct), 1) AS propina_pct_prom,
        ROUND(AVG(duracion_min), 1) AS duracion_min
    FROM silver_v0
    GROUP BY payment_type
    ORDER BY viajes DESC
""").show()

---
# FASE 4: Exploración libre y reflexión

## Sección 9: Escribe tu propio código

**Pregunta 9.1** - ¿Qué día de la semana genera más ingresos totales?

Pista: usa `dia_semana` y `F.sum('total_amount')`.

Respuesta: El día que genera más ingresos totales es el día 1.


In [ ]:
ingresos_por_dia = (
    df_s
    .groupBy('dia_semana')
    .agg(F.round(F.sum('total_amount'), 2).alias('total_ingresos'))
    .orderBy('total_ingresos', ascending=False)
)
ingresos_por_dia.show()


**Pregunta 9.2** - ¿A qué hora del dia se registran las tarifas más altas por milla?

Pista: usa `hora_inicio` y `F.avg('tarifa_por_milla')`.

Respuesta: La hora con tarifas más altas por milla es a las 04:00.


In [ ]:
tarifa_por_hora = (
    df_s
    .groupBy('hora_inicio')
    .agg(F.round(F.avg('tarifa_por_milla'), 2).alias('tarifa_promedio_milla'))
    .orderBy('tarifa_promedio_milla', ascending=False)
)
tarifa_por_hora.show(24)


## Sección 10: Reflexión final

### Pregunta 10.1 - Calidad de datos

In [ ]:
print('=' * 55)
print('   RESUMEN DEL PIPELINE BRONZE -> SILVER')
print('=' * 55)
print(f'   Registros Bronze:    {n_bronze:>12,}')
print(f'   Registros Silver:    {n_silver:>12,}')
print(f'   Descartados:         {n_bronze-n_silver:>12,}')
print(f'   Retencion:           {n_silver/n_bronze*100:>10.1f}%')
print(f'   Tiempo de pipeline:  {TIEMPO_PIPELINE:>10.2f}s')
print(f'   Tiempo Narrow:       {TIEMPO_NARROW:>10.2f}s')
print(f'   Tiempo Wide:         {TIEMPO_WIDE:>10.2f}s')
print(f'   Factor Wide/Narrow:  {TIEMPO_WIDE/TIEMPO_NARROW:>10.1f}x')
print('=' * 55)

**Escribe aquí tu respuesta:**

> **% de retención:** 93,9%
>
> **¿Es aceptable?:** Sí. Se mantiene casi todo el dataset y solo se descarta un 6,1%, que corresponde a registros con problemas como tarifas negativas, distancias en cero, pasajeros fuera de rango o viajes con horarios incorrectos. Para mi es aceptable porque mejora la calidad de los datos sin perder demasiado volumen.
>
> **¿Qué harías si el descarte supera el 20%?:** Revisaría primero qué regla está eliminando más registros. Después compararía con la fuente original para ver si hay un problema de ingesta o si los filtros quedaron muy estrictos. Si corresponde, corregiría datos, imputaría algunos valores o ajustaría las reglas antes de eliminar tantos registros.


### Pregunta 10.2 - Conexión con tu proyecto

**a)** *¿Cuántos Shuffles tiene el pipeline Bronze->Silver que construiste hoy? ¿En que transformación ocurren? ¿Eran necesarios o podrías haberlos evitado?*

**b)** *Vuelve a leer tu Fase 1. ¿Cambiarías alguna decisión de formato, ingesta o arquitectura de capas después de haber ejecutado este pipeline?*

**Escribe aquí tu respuesta:**

> **a) Shuffles identificados:** Identifiqué 1 Shuffle principal. Ocurre en la escritura de la capa Silver cuando se usa `.partitionBy('payment_type')`, porque Spark debe reorganizar los datos por tipo de pago. En el plan Bronze -> Silver las demás transformaciones son principalmente Narrow, como filtros, creación de columnas y selección. Este Shuffle era útil para dejar la tabla particionada, pero se podría evitar si no necesitáramos optimizar consultas por `payment_type`.
>
> **b) Decisión que cambiaría:** Después de ejecutar este pipeline, tendría más cuidado al elegir las columnas de partición. No usaría variables con mucha cardinalidad porque pueden generar muchos archivos pequeños y hacer más lento el procesamiento. Elegiría columnas simples y útiles para consulta, como fecha o una categoría estable.
>
> **Lo que mantendría igual:** Mantendría la arquitectura por capas Bronze y Silver, porque separa bien los datos crudos de los datos limpios. También mantendría Delta Lake, ya que permite control de versiones, auditoría y time travel, lo que sirve mucho si hay que revisar cambios o volver a una versión anterior.


---
## Cierre

In [ ]:
spark.stop()
print('SparkSession cerrada correctamente.')
print()
print('Guarda una copia en tu Google Drive: Archivo -> Guardar una copia en Drive')
print()
print('En la proxima sesion (5 de junio):')
print('  Modulo 4 - Servicios Gestionados de Nube para Big Data')
print('  Conectaremos este pipeline a GCP.')
print('  Será necesario crear tu cuenta GCP.')
print()
print('Pregunta abierta para la Clase 5 (Optimizacion y FinOps):')
print('  Hoy viste el Shuffle en .explain(). En la Clase 5 veremos:')
print('  -> Como diagnosticarlo en Spark UI')
print('  -> Como reducirlo con particionamiento y caching')
print('  -> Cuanto cuesta en dinero en Dataproc')

---

## Referencias

- Damji et al. (2020). *Learning Spark* (2nd ed., caps. 2-4). O'Reilly.
- Delta Lake Documentation. https://docs.delta.io/latest/
- Apache Spark Documentation. https://spark.apache.org/docs/latest/sql-programming-guide.html
- NYC TLC Trip Record Data. https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

---

*Magister en Data Science - Universidad del Desarrollo - Big Data y Cloud Computing - 2026*